# Qwen3.5-2B-Coder — validated Colab v1
This is the all-in-one convenience workflow. To save GPU credits, prefer `01_Prepare_Data_CPU.ipynb` followed by `02_Train_and_Publish_GPU.ipynb`. If you use this notebook, select an A100 40 GB or A100/H100 80 GB runtime and keep it connected until the final Hugging Face upload completes.

In [ ]:
from google.colab import userdata
import shutil, subprocess
from pathlib import Path
REPOSITORY_URL = 'https://github.com/SSusantAchary/coder-SFT.git'
PROJECT_ROOT = Path('/content/coder_SFT')
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
        raise RuntimeError(f'{PROJECT_ROOT} exists but is not a valid coder-SFT checkout')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
WORK_ROOT = Path('/content/qwen35-2b-coder')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
assert (PROJECT_ROOT / 'pyproject.toml').exists(), f'Missing project at {PROJECT_ROOT}'
disk = shutil.disk_usage('/content')
print(f'Colab disk: {disk.total / 2**30:.1f} GiB total, {disk.free / 2**30:.1f} GiB free')

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'uv'], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '-r', str(PROJECT_ROOT / 'requirements-colab.txt')], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '--no-build-isolation', '-r', str(PROJECT_ROOT / 'requirements-kernels.txt')], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '--no-deps', '-r', str(PROJECT_ROOT / 'requirements-accelerator.txt')], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '-e', str(PROJECT_ROOT)], check=True)
subprocess.run([sys.executable, '-c', 'import coder_sft; print(coder_sft.__version__)'], check=True)
print('Restart the runtime once if Torch or CUDA packages were replaced, then continue below.')

In [ ]:
import os, sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ['CODER_SFT_WORKDIR'] = str(WORK_ROOT)
hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Add a write-enabled HF_TOKEN to Colab Secrets before training.')
os.environ['HF_TOKEN'] = hf_token
from huggingface_hub import HfApi
HF_OWNER = HfApi(token=hf_token).whoami()['name']
HF_REPO_ID = f'{HF_OWNER}/Qwen3.5-2B-Coder-SFT'
HF_REPO_PRIVATE = True  # Set False only when you intentionally want a public model.
BASE = str(PROJECT_ROOT / 'configs/base.yaml')
DATA = str(PROJECT_ROOT / 'configs/data_v1.yaml')
HARDWARE = str(PROJECT_ROOT / 'configs/hardware.yaml')
STAGE1 = str(PROJECT_ROOT / 'configs/stage1_8k.yaml')
STAGE2 = str(PROJECT_ROOT / 'configs/stage2_repo_32k.yaml')
from coder_sft.config import load_config, write_resolved_config
from coder_sft.hardware import discover_hardware, resolve_hardware
from coder_sft.utils import environment_manifest, write_json
preflight = resolve_hardware(load_config(BASE, HARDWARE, STAGE1), 'stage1', discover_hardware())
(WORK_ROOT / 'reports').mkdir(exist_ok=True)
write_resolved_config(preflight, WORK_ROOT / 'reports/preflight_resolved.yaml')
write_json(environment_manifest(), WORK_ROOT / 'reports/preflight_environment.json')
print(preflight['resolved_hardware'])

## 1. Prepare datasets
Repository reconstruction is network/disk intensive. Data, repository caches, checkpoints, and exports remain on the Colab `/content` disk until the final Hub upload.

In [ ]:
# Optional eight-example, one-step GPU smoke test. Full preparation below replaces the tiny data file.
RUN_SMOKE = False
if RUN_SMOKE:
    subprocess.run(['prepare-data', '--config', BASE, '--config', DATA, '--limit', '8'], check=True)
    subprocess.run(['train-sft', '--config', BASE, '--config', HARDWARE, '--config', STAGE1, '--stage', 'stage1', '--resume', 'none', '--limit', '8', '--max-steps', '1', '--output-dir', str(WORK_ROOT / 'outputs/smoke')], check=True)

In [ ]:
subprocess.run(['prepare-data', '--config', BASE, '--config', DATA], check=True)
subprocess.run(['build-repo-context', '--config', BASE, '--config', DATA], check=True)

## 2. Baseline generation and Stage 1
Generated benchmark code is saved only; it is never executed in this notebook.

In [ ]:
REPORTS = WORK_ROOT / 'reports'; REPORTS.mkdir(exist_ok=True)
subprocess.run(['generate-eval', '--config', BASE, '--config', DATA, '--model', 'unsloth/Qwen3.5-2B', '--dataset', 'humaneval', '--output', str(REPORTS / 'base_humaneval.jsonl')], check=True)
subprocess.run(['generate-eval', '--config', BASE, '--config', DATA, '--model', 'unsloth/Qwen3.5-2B', '--dataset', 'mbpp', '--output', str(REPORTS / 'base_mbpp.jsonl')], check=True)

In [ ]:
subprocess.run(['train-sft', '--config', BASE, '--config', HARDWARE, '--config', STAGE1, '--stage', 'stage1', '--resume', 'auto'], check=True)
STAGE1_ADAPTER = WORK_ROOT / 'outputs/q35-2b-coder-s1-8k/final_adapter'
subprocess.run(['generate-eval', '--config', BASE, '--config', DATA, '--model', str(STAGE1_ADAPTER), '--dataset', 'humaneval', '--output', str(REPORTS / 'stage1_humaneval.jsonl')], check=True)
subprocess.run(['generate-eval', '--config', BASE, '--config', DATA, '--model', str(STAGE1_ADAPTER), '--dataset', 'mbpp', '--output', str(REPORTS / 'stage1_mbpp.jsonl')], check=True)

## 3. Memory profile and gated Stage 2
Score Stage 1 externally, place `base.json` and `stage1.json` in the local reports directory, then run this cell. It creates the mandatory gate, profiles memory, and starts Stage 2 only if the gate passes.

In [ ]:
STAGE1_ADAPTER = WORK_ROOT / 'outputs/q35-2b-coder-s1-8k/final_adapter'
BASE_SCORES = REPORTS / 'base.json'
STAGE1_SCORES = REPORTS / 'stage1.json'
if not BASE_SCORES.exists() or not STAGE1_SCORES.exists():
    raise FileNotFoundError('External EvalPlus scores are required: reports/base.json and reports/stage1.json')
subprocess.run(['compare-runs', '--baseline', str(BASE_SCORES), '--candidate', str(STAGE1_SCORES), '--stage', 'stage1', '--output', str(REPORTS / 'stage1_gate.json')], check=True)
subprocess.run(['profile-memory', '--config', BASE, '--config', HARDWARE, '--config', STAGE2], check=True)
subprocess.run(['train-sft', '--config', BASE, '--config', HARDWARE, '--config', STAGE2, '--stage', 'stage2', '--adapter', str(STAGE1_ADAPTER), '--resume', 'auto'], check=True)

## 4. Final export and Hugging Face upload
Run this only after Stage 2 completes. It verifies adapter lineage and merged-model equivalence, creates the Hub repository if needed, then uploads the merged BF16 model and the LoRA adapter.

In [ ]:
STAGE2_ADAPTER = WORK_ROOT / 'outputs/q35-2b-coder-s2-repo/final_adapter'
FINAL_EXPORT = WORK_ROOT / 'exports/q35-coder-merged'
publish = ['export-model', '--adapter', str(STAGE2_ADAPTER), '--output', str(FINAL_EXPORT), '--format', 'merged', '--hub-model-id', HF_REPO_ID]
if not HF_REPO_PRIVATE: publish.append('--public')
subprocess.run(publish, check=True)
print(f'Published final model: https://huggingface.co/{HF_REPO_ID}')